In [ ]:
#Professor: Limite a 150 skills porque se não é ilegível. Fiz com vários grupos mas visualmente só acho relevante 2/3

mlb = MultiLabelBinarizer()
skills_matrix = pd.DataFrame(mlb.fit_transform(df_skills['skills_list']), columns=mlb.classes_)

min_occurrence = 50 
frequent_skills = skills_matrix.columns[skills_matrix.sum() > min_occurrence]

if len(frequent_skills) > 150:
    frequent_skills = skills_matrix.sum().sort_values(ascending=False).head(150).index

matrix_to_cluster = skills_matrix[frequent_skills].T

n_clusters = 2
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
clusters = kmeans.fit_predict(matrix_to_cluster)

pca = PCA(n_components=2)
reduced_data = pca.fit_transform(matrix_to_cluster)

plt.figure(figsize=(16, 12))
scatter_df = pd.DataFrame({
    'x': reduced_data[:, 0],
    'y': reduced_data[:, 1],
    'label': frequent_skills,
    'cluster': clusters
})

sns.scatterplot(data=scatter_df, x='x', y='y', hue='cluster', palette='viridis', s=100, alpha=0.7)

texts = []
for i in range(len(scatter_df)):
    texts.append(plt.text(scatter_df.x[i], scatter_df.y[i], scatter_df.label[i], fontsize=11))

adjust_text(texts, force_points=0.2, force_text=0.2, expand_points=(1.2, 1.2))

plt.title('K-Means Clustering - Filtragem e Legibilidade')
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

In [ ]:
skills_as_sentences = df_skills['skills_list'].apply(lambda x: " ".join(x))
vectorizer = CountVectorizer()
doc_term_matrix = vectorizer.fit_transform(skills_as_sentences)

n_topics = 5
lda = LatentDirichletAllocation(n_components=n_topics, random_state=42)
lda.fit(doc_term_matrix)

words = vectorizer.get_feature_names_out()
fig, axes = plt.subplots(1, n_topics, figsize=(25, 12), sharex=False)
axes = axes.flatten()

for topic_idx, topic in enumerate(lda.components_):
    top_features_ind = topic.argsort()[:-11:-1]
    top_features = [words[i] for i in top_features_ind]
    weights = topic[top_features_ind]

    ax = axes[topic_idx]
    ax.barh(top_features, weights, color='steelblue')
    ax.set_title(f'Perfil {topic_idx}', fontdict={'fontsize': 18})
    ax.invert_yaxis()
    ax.tick_params(axis='both', which='major', labelsize=14)

plt.tight_layout()
plt.savefig('lda_viz.png')

for topic_idx, topic in enumerate(lda.components_):
    top_words = [words[i] for i in topic.argsort()[:-11:-1]]
    print(f"Perfil {topic_idx}: {', '.join(top_words)}")